# 03 · Revisión interna y métricas

Para preparar la reunión con Martin y decidir qué prompts usar. Solo lee datos (salvo los veredictos de la sección 4).

1. Ranking de **todas** las versiones de prompt en A1 (anotación a ciegas)
2. Métricas de los 4 prompts y de la **moda de los 3 mejores**, en A1 y en A2
3. Explicación de las métricas
4. Persona vs modelos, foto por foto, con **veredicto** (*my error / model error / ask Martin*)
5. Registros posiblemente duplicados (mismo día y lugar)

**Cuidado con el sesgo:**
- A1 se anotó a ciegas: es la referencia para medir. A2 se anotó con los modelos a la vista: su acuerdo sale inflado.
- El ranking y la moda se eligen con las mismas fotos de A1 con las que se evalúan: la cifra sale algo optimista. Hace falta un set de validación nuevo (notebook 02, sección 3).

| Notebook | Para qué |
|---|---|
| `01_descarga_y_modelos` | bajar registros de iNaturalist y correr el triaje y los 4 prompts |
| `02_anotacion` | anotar: asistida (elegir el mejor valor de los modelos), revisión con letra y recuadro, o a ciegas |
| `03_revision_y_metricas` | revisión interna: ranking de prompts, moda, métricas, persona vs modelos, duplicados |
| `04_revision_con_martin` | lo que se le muestra a Martin: qué hicimos, resultados, fotos, preguntas y el Excel |
| `05_entrega_excel` | generar y revisar el Excel de entrega |

La versión oficial de cada notebook está en GitHub (`notebooks/`, sin resultados). Para trabajar: abrirlo desde GitHub en Colab, *Archivo → Guardar una copia en Drive* y moverla a `MyDrive/sepia-inaturalist/notebooks/` (con la fecha en el nombre).

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'

In [ ]:
# Corridas (cada una en su carpeta de Drive). A1: anotacion a ciegas. A2: anotacion asistida.
cfg_a1 = sepia.load_config('configs/pilot_a1.yaml')
cfg_a2 = sepia.load_config('configs/a2_test.yaml')
RUNS = [cfg_a1, cfg_a2]

In [ ]:
from sepia.disagree import (rank_versions, classification_metrics, plot_confusion, count_agreement,
                            DisagreementApp, verdict_summary)

## 1. Ranking de todas las versiones en A1

Evalúa todas las versiones con resultados (v1 a v14) sobre las mismas fotos anotadas, y las ordena por el promedio de acuerdo en conteo, lado, erosión y calidad. Ojo: `side_agreement` cuenta `unclear` como error; mirar también `side_agreement_when_decided`.

In [ ]:
all_ranking, _ = rank_versions(cfg_a1, n_top=5, save=False)

## 2. Los 4 prompts y la moda de los 3 mejores

La **moda** toma, por foto y atributo, el valor que dan al menos 2 de los 3 mejores; si los 3 difieren, `unclear` (en el conteo: la foto se salta). Los 3 mejores se eligen entre los 4 prompts con el ranking de A1.

In [ ]:
ranking, _ = rank_versions(cfg_a1, VERSIONS, n_top=3, save=False)
TOP3 = list(ranking.index[:3])
print('3 mejores (A1):', TOP3)

### 2a. A1 (a ciegas)

In [ ]:
count_agreement(cfg_a1, VERSIONS, mode_of=TOP3);
summary_a1, per_class_a1, mats_a1 = classification_metrics(cfg_a1, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a1, 'side');
plot_confusion(mats_a1, 'erosion');
plot_confusion(mats_a1, 'photo_quality');

### 2b. A2 (asistida, solo referencia)

In [ ]:
count_agreement(cfg_a2, VERSIONS, mode_of=TOP3);
summary_a2, per_class_a2, mats_a2 = classification_metrics(cfg_a2, VERSIONS, save=False, mode_of=TOP3)

In [ ]:
plot_confusion(mats_a2, 'side');
plot_confusion(mats_a2, 'erosion');
plot_confusion(mats_a2, 'photo_quality');

## 3. Qué mide cada métrica

Mirando una clase (por ejemplo **dorsal**): VP = anotado dorsal y el modelo dice dorsal; FP = el modelo dice dorsal pero no lo es; FN = anotado dorsal y el modelo dice otra cosa o `unclear`.

- **Exactitud (accuracy)** = acertadas / todas (un `unclear` cuenta como error). **accuracy_when_decided**: solo donde el modelo decidió. **Cobertura**: fracción donde decidió.
- **Precisión** = VP / (VP + FP): cuando dice dorsal, ¿cuántas veces lo es?
- **Recall** = VP / (VP + FN): de los dorsales, ¿cuántos encontró?
- **F1** = 2·P·R / (P + R): alta solo si las dos lo son.
- **macro F1** y **exactitud balanceada**: cada clase pesa lo mismo.
- **Kappa de Cohen** (pendiente de agregar): acuerdo descontando el azar; para la erosión, kappa ponderado.

## 4. Persona vs modelos, con veredicto

Todas las fotos anotadas de A1 y A2 (filtro *Run*). Para cada atributo con desacuerdo, marcar **My error**, **Model error** o **Unsure / ask Martin**, y *Save* (la foto queda revisada). Los veredictos se guardan en `reviews/` de cada corrida; no cambian ninguna anotación.

In [ ]:
DisagreementApp(RUNS, VERSIONS, ANNOTATOR, view='any_model_differs', mode_of=TOP3).show()

In [ ]:
# Resumen de veredictos: registros a corregir en el notebook 02 y preguntas para Martin
for c in RUNS:
    print('=====', c.base_dir.split('/')[-1])
    verdict_summary(c, ANNOTATOR)

## 5. Registros posiblemente duplicados

Pares de registros del **mismo día** y a ≤ 2 km (primero se filtra por día, después se calcula la distancia). Solo son candidatos: si es el mismo hueso, se marca en la anotación (*Same bone as*).

In [ ]:
from sepia.duplicates import load_runs, candidate_pairs, candidate_groups
obs_all = load_runs(cfg_a1, ['pilot_a1', 'a2_test'])
pairs = candidate_pairs(obs_all, max_km=2)
groups = candidate_groups(pairs)
print(f"{len(pairs)} pares · {len(groups)} grupos")
display(groups)
display(pairs)